# 표 데이터 긁어오기

이 노트북은 PMC 문서에서 테이블 내용을 제대로 파싱하는지 테스트합니다.

## 변경사항
- `pmc_parsing.py`에 테이블 내용 파싱 로직 추가
- 테이블의 헤더, 데이터 행, 셀 정보를 추출
- 수식(`<sup>`, `<sub>`, `<inline-formula>` 등) 처리 포함

## 테스트 문서
- PMCID: PMC12529775
- 테이블 포함 여부: ✓

아래 셀을 실행하면:
1. PMC OAI API에서 테스트 문서를 가져옵니다
2. `extract_article_info()` 함수로 파싱합니다
3. 테이블 정보와 내용을 출력합니다
4. 결과를 `test_table_output.json` 파일로 저장합니다

In [1]:
import sys
from pathlib import Path

# 프로젝트 루트 경로를 Python 경로에 추가
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

import requests
import xml.etree.ElementTree as ET
import json

# 모듈 리로드 (코드 수정 후 다시 테스트할 때 필요)
import importlib
from rag.etl.step01_ingest.pmc_ingest_common import pmc_parsing
importlib.reload(pmc_parsing)
from rag.etl.step01_ingest.pmc_ingest_common.pmc_parsing import extract_article_info

# PMC OAI API 엔드포인트
PMC_OAI_ENDPOINT = "https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi"

# 테스트할 PMCID (테이블이 포함된 문서)
test_pmcid = "PMC12529775"

# PMC OAI API로 문서 가져오기
params = {
    "verb": "GetRecord",
    "identifier": f"oai:pubmedcentral.nih.gov:{test_pmcid.replace('PMC', '')}",
    "metadataPrefix": "pmc"
}

print(f"테스트 문서: {test_pmcid}")
print(f"API 요청 중...")

response = requests.get(PMC_OAI_ENDPOINT, params=params, timeout=30)
response.raise_for_status()

print(f"응답 받음 (길이: {len(response.text)} bytes)")

# XML 파싱
root = ET.fromstring(response.content)

# GetRecord 응답에서 record 찾기
ns_oai = {"oai": "http://www.openarchives.org/OAI/2.0/"}
record = root.find(".//oai:record", ns_oai)

if record is None:
    print("❌ record를 찾을 수 없습니다.")
else:
    print("✓ record 찾음")
    
    # extract_article_info 호출
    article_info = extract_article_info(record)
    
    if article_info:
        print(f"\n✓ 문서 파싱 완료")
        print(f"  제목: {article_info.get('title', '')[:100]}...")
        print(f"  테이블 수: {len(article_info.get('table_captions', []))}")
        
        # 테이블 정보 출력
        for i, table in enumerate(article_info.get('table_captions', [])):
            print(f"\n=== 테이블 {i+1} ===")
            print(f"  ID: {table.get('id')}")
            print(f"  Label: {table.get('label')}")
            print(f"  Caption: {table.get('caption', '')[:100]}...")
            print(f"  Page URL: {table.get('page_url')}")
            
            # 테이블 내용 확인
            content = table.get('content')
            if content:
                print(f"\n  ✓ 테이블 내용 파싱됨!")
                
                # Structured (JSON) 확인
                structured = content.get('structured', {})
                print(f"    헤더 행 수: {len(structured.get('headers', []))}")
                print(f"    데이터 행 수: {len(structured.get('rows', []))}")
                
                # 헤더 출력
                if structured.get('headers'):
                    print(f"\n  [JSON - 헤더]")
                    for h_idx, header_row in enumerate(structured['headers']):
                        header_texts = [cell['text'] for cell in header_row]
                        print(f"    행 {h_idx+1}: {header_texts}")
                
                # 데이터 행 일부 출력 (최대 3행)
                if structured.get('rows'):
                    print(f"\n  [JSON - 데이터 행 (최대 3행)]")
                    for r_idx, row in enumerate(structured['rows'][:3]):
                        row_texts = [cell['text'] for cell in row]
                        print(f"    행 {r_idx+1}: {row_texts}")
                
                # Markdown 출력
                markdown = content.get('markdown', '')
                if markdown:
                    print(f"\n  [Markdown]")
                    print("    " + "\n    ".join(markdown.split('\n')[:6]) + "...")
                
                # Text 출력
                text = content.get('text', '')
                if text:
                    print(f"\n  [Text (검색용)]")
                    print(f"    {text[:200]}...")
                
            else:
                print(f"\n  ❌ 테이블 내용이 파싱되지 않음")
        
        # JSON 저장 (확인용)
        output_file = "test_table_output.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(article_info, f, indent=2, ensure_ascii=False)
        print(f"\n✓ 결과를 '{output_file}'에 저장했습니다.")
    else:
        print("❌ 문서 파싱 실패")

테스트 문서: PMC12529775
API 요청 중...
응답 받음 (길이: 108672 bytes)
✓ record 찾음
  [PMC HTML Fetch Error] PMC12529775: 403 Client Error: Forbidden for url: https://pmc.ncbi.nlm.nih.gov/articles/PMC12529775/

✓ 문서 파싱 완료
  제목: Unraveling the Catalytic Mechanism of β‑Cyclodextrin
in the Vitamin D Formation...
  테이블 수: 1

=== 테이블 1 ===
  ID: tbl1
  Label: table1
  Caption: Ratio Between the Rate Constant for
the Encapsulated System, [e], and the Free System, [f]...
  Page URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC12529775/#tbl1

  ✓ 테이블 내용 파싱됨!
    헤더 행 수: 0
    데이터 행 수: 0

✓ 결과를 'test_table_output.json'에 저장했습니다.


# 2. 수식 이미지 삽입 인식하기

## chandra
...pod띄워서 해야하자너..?